In [3]:
import numpy as np
from scipy.stats import wilcoxon

# Seed results
# Order: seed 42, 123, 7, 21, 99, 555

raw_text = {
    "Accuracy":      [0.9227, 0.9300, 0.9415, 0.9342, 0.9310, 0.9289],
    "Precision":     [0.8603, 0.8468, 0.8588, 0.8492, 0.8619, 0.8462],
    "Recall":        [0.8243, 0.8787, 0.9163, 0.8954, 0.8619, 0.8745],
    "F1":            [0.8419, 0.8624, 0.8866, 0.8717, 0.8619, 0.8601],
    "ROC-AUC":       [0.9592, 0.9662, 0.9666, 0.9629, 0.9622, 0.9618],
    "Avg Precision": [0.8843, 0.8796, 0.8750, 0.8812, 0.8697, 0.8841],
}

srl_markup = {
    "Accuracy":      [0.9342, 0.9331, 0.9363, 0.9363, 0.9321, 0.9342],
    "Precision":     [0.8520, 0.8302, 0.8371, 0.8504, 0.8372, 0.8411],
    "Recall":        [0.8912, 0.9205, 0.9247, 0.9038, 0.9038, 0.9079],
    "F1":            [0.8712, 0.8730, 0.8787, 0.8763, 0.8692, 0.8732],
    "ROC-AUC":       [0.9687, 0.9632, 0.9649, 0.9700, 0.9651, 0.9610],
    "Avg Precision": [0.8892, 0.8686, 0.8815, 0.8972, 0.9005, 0.8747],
}

no_contrastive = {
    "Accuracy":      [0.9342, 0.9310, 0.9279, 0.9331, 0.9289, 0.9279],
    "Precision":     [0.8577, 0.8264, 0.8400, 0.8302, 0.8577, 0.8455],
    "Recall":        [0.8828, 0.9163, 0.8787, 0.9205, 0.8577, 0.8703],
    "F1":            [0.8701, 0.8690, 0.8589, 0.8730, 0.8577, 0.8577],
    "ROC-AUC":       [0.9622, 0.9599, 0.9630, 0.9585, 0.9689, 0.9643],
    "Avg Precision": [0.8600, 0.8634, 0.8750, 0.8438, 0.8934, 0.8577],
}

no_cross_attention = {
    "Accuracy":      [0.9300, 0.9331, 0.9342, 0.9383, 0.9289, 0.9289],
    "Precision":     [0.8554, 0.8405, 0.8667, 0.8571, 0.8406, 0.8519],
    "Recall":        [0.8661, 0.9038, 0.8703, 0.9038, 0.8828, 0.8661],
    "F1":            [0.8607, 0.8710, 0.8685, 0.8798, 0.8612, 0.8589],
    "ROC-AUC":       [0.9698, 0.9671, 0.9674, 0.9660, 0.9666, 0.9646],
    "Avg Precision": [0.8994, 0.8908, 0.8988, 0.8988, 0.8830, 0.8723],
}

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Avg Precision"]

comparisons = [
    ("SRL Markup vs Raw Text",              srl_markup, raw_text),
    ("SRL Markup vs No Contrastive Loss",   srl_markup, no_contrastive),
    ("SRL Markup vs No Cross-Attention",    srl_markup, no_cross_attention),
]

# Run tests 

print("=" * 80)
print("WILCOXON SIGNED-RANK TEST RESULTS")
print("Paired, two-sided | n=6 seeds | α=0.05")
print("=" * 80)

all_results = {}

for comp_name, variant_a, variant_b in comparisons:
    print(f"\n{'─' * 80}")
    print(f"  {comp_name}")
    print(f"{'─' * 80}")
    print(f"  {'Metric':<18} {'Mean A':>8} {'Mean B':>8} {'Diff':>8} {'W-stat':>8} {'p-value':>10} {'Sig?':>6}")
    print(f"  {'-'*18} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*10} {'-'*6}")

    comp_results = {}
    for metric in metrics:
        a = np.array(variant_a[metric])
        b = np.array(variant_b[metric])
        diff = a - b

        # Wilcoxon requires non-zero differences
        # If all differences are zero, test is not applicable
        if np.all(diff == 0):
            print(f"  {metric:<18} {np.mean(a):>8.4f} {np.mean(b):>8.4f} "
                  f"{'0.0000':>8} {'N/A':>8} {'N/A':>10} {'N/A':>6}")
            comp_results[metric] = {"stat": None, "p": None, "mean_diff": 0.0}
            continue

        try:
            stat, p = wilcoxon(a, b, alternative="two-sided")
        except ValueError as e:
            # Raised when all differences are zero
            print(f"  {metric:<18} — wilcoxon error: {e}")
            continue

        sig = "✓" if p < 0.05 else "✗"
        mean_diff = np.mean(diff)
        print(f"  {metric:<18} {np.mean(a):>8.4f} {np.mean(b):>8.4f} "
              f"{mean_diff:>+8.4f} {stat:>8.1f} {p:>10.4f} {sig:>6}")

        comp_results[metric] = {"stat": stat, "p": p, "mean_diff": mean_diff}

    all_results[comp_name] = comp_results

# 95% Confidence intervals on mean differences

print("=" * 80)
print("95% BOOTSTRAP CONFIDENCE INTERVALS ON MEAN F1 DIFFERENCES")
print("(n=6 seeds, 10,000 bootstrap iterations)")
print("=" * 80)

np.random.seed(42)
N_BOOT = 10_000

for comp_name, variant_a, variant_b in comparisons:
    a = np.array(variant_a["F1"])
    b = np.array(variant_b["F1"])
    diff = a - b
    boot_means = [
        np.mean(np.random.choice(diff, size=len(diff), replace=True))
        for _ in range(N_BOOT)
    ]
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    print(f"  {comp_name:<45} mean diff = {np.mean(diff):+.4f}  "
          f"95% CI [{lo:+.4f}, {hi:+.4f}]")


WILCOXON SIGNED-RANK TEST RESULTS
Paired, two-sided | n=6 seeds | α=0.05

────────────────────────────────────────────────────────────────────────────────
  SRL Markup vs Raw Text
────────────────────────────────────────────────────────────────────────────────
  Metric               Mean A   Mean B     Diff   W-stat    p-value   Sig?
  ------------------ -------- -------- -------- -------- ---------- ------
  Accuracy             0.9344   0.9314  +0.0030      4.0     0.2188      ✗
  Precision            0.8413   0.8539  -0.0125      1.0     0.0625      ✗
  Recall               0.9087   0.8752  +0.0335      0.0     0.0312      ✓
  F1                   0.8736   0.8641  +0.0095      3.0     0.1562      ✗
  ROC-AUC              0.9655   0.9632  +0.0023      7.0     0.5625      ✗
  Avg Precision        0.8853   0.8790  +0.0063      7.0     0.5625      ✗

────────────────────────────────────────────────────────────────────────────────
  SRL Markup vs No Contrastive Loss
─────────────────────